# 04 · Cloud pass: Jev and the open decision models on the VM, two question-set versions

The open models run on the Terraform VM (`infra/gcp`, two NVIDIA L4, zone chosen where capacity exists). Which models, and on which ports, is read from the VM's metadata at run time by `goldrails_bench.endpoints`; nothing is stored in the repo or in `.env`. The tunnels come from ``make tunnel``. Bedrock Guardrails (InvokeGuardrailChecks) is the managed baseline when AWS credentials are present; it answers only the questions that map to its categories and the rest are recorded as unanswered. Jev is the hosted API. Every system gets the identical question set through `SystemOneClient`.

Rows: the F1 tune split of the pilot (20 rows). Question sets: v1 and v2 of `f1-bedrock5` and `f1-ailuminate12`. This is the plumbing check for the cloud path; the scaled run writes receipts to disk with the harness.

In [ ]:
import os, time, json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from goldrails_bench.data import load_rows
from goldrails_bench.bedrock import BedrockChecksClient
from goldrails_bench import question_sets
from goldrails_bench.systemone import SystemOneClient, state_of
from goldrails_bench.runner import run_matrix

load_dotenv(find_dotenv(usecwd=True))
rows = load_rows("F1", "tune")     # GOLDRAILS_DATA picks local samples today, hf:raxITLabs/gold-rails later; nothing here changes
from goldrails_bench.endpoints import resolve_models, describe
info = describe(); print(info["instance"], info["status"], info["machine_type"], "external_ip=", info["external_ip"])
served = resolve_models(mode="tunnel")   # served models, from the VM's metadata; needs `make tunnel`
systems = {"jev-1.13.0": SystemOneClient("jev-1.13.0", model="jev-1.13.0", identity={"model": "jev-1.13.0", "provider": "api.typesafe.ai"})}
systems.update({m["name"]: SystemOneClient(m["name"], base_url=m["url"], model=m["model"], identity=m["identity"]) for m in served})   # identity = checkpoint ref + revision, hashed into every ledger record
gpu_of = {m["name"]: m["gpu"] for m in served}   # systems sharing a card run one after another; cards and APIs in parallel
# Managed baseline: Bedrock Guardrails via InvokeGuardrailChecks, asked the same question sets (mapped to its categories).
# Needs AWS_PROFILE/AWS_REGION in .env and a live SSO session; if the probe fails the run proceeds without it and says so.
try:
    bedrock = BedrockChecksClient()
    probe_call = bedrock.ask({"role": "user", "text": "hello there"}, {"hate": {}})
    if probe_call.ok: systems["bedrock-checks"] = bedrock
    else: print("bedrock unavailable:", probe_call.error)
except Exception as e:
    print("bedrock unavailable:", type(e).__name__, e)
qsets = {f"{v}-{name}": question_sets.load(v, name) for v in ("v1", "v2") for name in question_sets.available(v)}
merged = {"all": {"questions": question_sets.merge(qsets)}}   # one call per row carries every question set; split() puts answers back
len(rows), list(systems), list(qsets)

## Reachability

One tiny call per system before the loop, so a dead tunnel shows up here and not as 20 failures later.

In [ ]:
probe = {"q": {"type": "noul", "instructions": "Is this text in English?", "criteria": {"true": "English", "false": "not English"}}}
for name, cl in systems.items():
    r = cl.ask({"role": "user", "text": "hello there"}, probe)
    print(f"{name:12s} ok={r.ok} model={r.model} {r.latency_s:.2f}s {r.error or ''}")


## Run

All four question sets go with each row (System One evaluates questions independently and in parallel; checked answer-identical against separate calls). Jev takes them in one 42-question call; local servers get chunks of at most 16 per call, because a server's peak GPU memory grows with questions per call and three servers share one card. Hosted APIs and each GPU run in parallel; models that share a GPU run one after another. Eight requests in flight against Jev, two against each local server. Nothing is retried. Serial per-call latency is a separate small pass below, never taken from this loop.


In [ ]:
t0 = time.perf_counter()
raw = run_matrix(systems, merged, rows, workers={"jev-1.13.0": 8}, gpu_of=gpu_of,
                 results_path="../results/pilot-cloud-pass.jsonl")   # every finished row is appended; a rerun resumes
wall = time.perf_counter() - t0
calls = []
for c in raw:                                   # explode each merged call back into one row per question set
    for qname, answers in (question_sets.split(c["answers"]) if c["ok"] else {q: None for q in qsets}).items():
        calls.append({**c, "question_set": qname, "answers": answers})
df = pd.DataFrame(calls)
print(f"{len(raw)} calls ({len(df)} system x question-set x row results) in {wall:.0f}s wall")
df.groupby(["system"]).agg(n=("ok","size"), ok=("ok","sum"), p50_s=("latency_s","median"), p95_s=("latency_s", lambda s: s.quantile(.95)))

## Serial latency, one request at a time

Five rows, one question set, concurrency 1, network included. This is the number a latency table may use; the throughput pass above is not.


In [ ]:
lat = pd.DataFrame(run_matrix(systems, {"v1-f1-bedrock5": qsets["v1-f1-bedrock5"]}, rows[:5], serial=True, progress=lambda *_: None))
lat.groupby("system").agg(n=("ok","size"), ok=("ok","sum"), p50_s=("latency_s","median"), max_s=("latency_s","max"))

## Max category probability against the label, and the intent signal in v2

`max_p` is the highest hazard indicator. In v2, `intent` is the real-world-intent Noul; a benign-but-scary row should show high `max_p` and low `intent`.

In [ ]:
def summarise(row):
    a = row["answers"] or {}
    nouls = {k: v["noul"] for k, v in a.items() if isinstance(v, dict) and v.get("type") == "noul" and k != "intent_real_world"}
    return pd.Series({"max_p": round(max(nouls.values()), 2) if nouls else None,
                      "intent": round(a["intent_real_world"]["noul"], 2) if "intent_real_world" in a else None,
                      "severity": round(a["severity"]["score"], 2) if "severity" in a else None})
out = pd.concat([df, df.apply(summarise, axis=1)], axis=1)
out.pivot_table(index=["subtask","expected","id"], columns=["question_set","system"], values="max_p")

In [ ]:
# Over-refusal rows only: did v2's carve-outs and intent Noul change the picture?
orb = out[out["subtask"]=="over_refusal"]
orb.pivot_table(index=["id"], columns=["question_set","system"], values=["max_p","intent"], aggfunc="first")

## Errors, if any

In [ ]:
df[~df["ok"]][["question_set","system","id","error"]]